# 数据概览 — Overfit 实验

为任务进展汇报 PPT 生成数据部分的图件和统计信息。

数据来源：2024 年日本 KiK-Net 强震观测数据  
全集：1004 个地震事件  
Overfit 子集：16 个事件（通过 `select_diverse_event_ids` 最大化震级/位置多样性选出）

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({
    'font.size': 12,
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
})

HDF5_PATH = '/tmp/japan_build_test3/japan_2024.hdf5'
TRAIN_CSV = '/tmp/japan_build_test3/train_ev_899e21fd.csv'
TEST_CSV  = '/tmp/japan_build_test3/test_ev_899e21fd.csv'
OUT_DIR   = Path('ppt_figures')
OUT_DIR.mkdir(exist_ok=True)

## 1. 加载数据

In [ ]:
# 读取 overfit 子集 CSV（train/test 完全相同的 16 个事件）
train_csv = pd.read_csv(TRAIN_CSV)
overfit_events = train_csv.drop_duplicates(subset='EVENT', keep='first').copy()
overfit_event_names = set(overfit_events['EVENT'].unique())
print(f'Overfit 子集: {len(overfit_event_names)} 个事件, {len(train_csv)} 条台站记录')

# 读取 HDF5 全集元数据
f = h5py.File(HDF5_PATH, 'r')
em = f['metadata']['event_metadata']
full_events = pd.DataFrame({
    'EVENT':     [x.decode() if isinstance(x, bytes) else x for x in em['EVENT'][()]],
    'Magnitude': em['Magnitude'][()],
    'Latitude':  em['Latitude'][()],
    'Longitude': em['Longitude'][()],
    'DEPTH':     em['DEPTH'][()],
    'N_Stations': em['N_Stations'][()],
})
full_events['is_overfit'] = full_events['EVENT'].isin(overfit_event_names)
print(f'全集: {len(full_events)} 个事件')

# 读取台站级元数据
sm = f['metadata']['station_metadata']
station_meta = pd.DataFrame({
    'EVENT':    [x.decode() if isinstance(x, bytes) else x for x in sm['EVENT'][()]],
    'station_lat': sm['station_lat'][()],
    'station_lon': sm['station_lon'][()],
    'station_code': [x.decode() if isinstance(x, bytes) else x for x in sm['station_code'][()]],
    'pga_mps2': sm['pga_norm_resampled_mps2'][()],
    'hypo_dist_km': sm['hypocentral_distance_km'][()],
    'stalta_ratio': sm['stalta_ratio_at_pick'][()],
})
station_meta['is_overfit'] = station_meta['EVENT'].isin(overfit_event_names)
station_meta['log10_pga'] = np.log10(station_meta['pga_mps2'].clip(lower=1e-10))

## 2. 文字统计摘要（直接放到 PPT 里）

In [ ]:
ov = full_events[full_events['is_overfit']]
stations_per_ov = train_csv.groupby('EVENT').size()

print('='*60)
print('PPT 文字要点')
print('='*60)
print(f'数据来源: 日本 KiK-Net 强震观测网, 2024 年')
print(f'全集: {len(full_events)} 个地震事件, 震级 M{full_events["Magnitude"].min():.1f}–{full_events["Magnitude"].max():.1f}')
print(f'Overfit 子集: {len(ov)} 个事件 (通过最大化震级-位置多样性选出)')
print(f'  震级范围: M{ov["Magnitude"].min():.1f}–{ov["Magnitude"].max():.1f}')
print(f'  震源深度: {ov["DEPTH"].min():.0f}–{ov["DEPTH"].max():.0f} km')
print(f'  每事件台站数: {stations_per_ov.min()}–{stations_per_ov.max()}, 中位数 {stations_per_ov.median():.0f}')
print(f'  训练/验证集: 相同 (overfit 模式, train=val)')
print(f'采样率: 100 Hz, 波形通道: 3 (NS, EW, UD)')

## 3. 图 1 — 震级分布（全集 vs overfit 子集）

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

bins = np.arange(2.75, 8.25, 0.5)
ax.hist(full_events['Magnitude'], bins=bins, color='steelblue', alpha=0.7,
        edgecolor='white', label=f'Full dataset (n={len(full_events)})')
ax.hist(ov['Magnitude'], bins=bins, color='tomato', alpha=0.85,
        edgecolor='white', label=f'Overfit subset (n={len(ov)})')

ax.set_xlabel('Magnitude')
ax.set_ylabel('Number of events')
ax.set_title('Magnitude Distribution: Full Dataset vs. Overfit Subset')
ax.legend()
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
fig.savefig(OUT_DIR / '01_magnitude_distribution.png')
plt.show()

## 4. 图 2 — 震中分布地图（全集灰底 + overfit 子集高亮）

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))

# 全集：灰色小点
rest = full_events[~full_events['is_overfit']]
ax.scatter(rest['Longitude'], rest['Latitude'], s=8, c='lightgrey',
           alpha=0.5, label=f'Other events (n={len(rest)})', zorder=1)

# Overfit 子集：按震级着色
sc = ax.scatter(ov['Longitude'], ov['Latitude'],
                s=80, c=ov['Magnitude'], cmap='YlOrRd', edgecolors='black',
                linewidths=0.8, vmin=5.0, vmax=7.0,
                label=f'Overfit subset (n={len(ov)})', zorder=2)
cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label('Magnitude')

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Epicenter Map — Japan 2024')
ax.legend(loc='lower left', fontsize=10)
ax.set_aspect(1.2)
fig.savefig(OUT_DIR / '02_epicenter_map.png')
plt.show()

## 5. 图 3 — 震源深度 vs 震级

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))

ax.scatter(full_events[~full_events['is_overfit']]['Magnitude'],
           full_events[~full_events['is_overfit']]['DEPTH'],
           s=10, c='lightgrey', alpha=0.4, label='Other events')
ax.scatter(ov['Magnitude'], ov['DEPTH'],
           s=60, c='tomato', edgecolors='black', linewidths=0.6,
           label='Overfit subset', zorder=2)

ax.set_xlabel('Magnitude')
ax.set_ylabel('Depth (km)')
ax.invert_yaxis()
ax.set_title('Depth vs. Magnitude')
ax.legend()
fig.savefig(OUT_DIR / '03_depth_vs_magnitude.png')
plt.show()

## 6. 图 4 — 每事件台站数分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 全集
axes[0].hist(full_events['N_Stations'], bins=40, color='steelblue',
             edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Stations per event')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Full dataset (n={len(full_events)})')
axes[0].axvline(full_events['N_Stations'].median(), color='navy',
                ls='--', label=f'median={full_events["N_Stations"].median():.0f}')
axes[0].legend()

# Overfit 子集
axes[1].barh(range(len(stations_per_ov)),
             stations_per_ov.sort_values().values,
             color='tomato', edgecolor='white')
sorted_events = stations_per_ov.sort_values()
event_labels = []
for ev in sorted_events.index:
    mag = ov.loc[ov['EVENT']==ev, 'Magnitude'].values[0]
    event_labels.append(f'M{mag:.1f}')
axes[1].set_yticks(range(len(sorted_events)))
axes[1].set_yticklabels(event_labels, fontsize=9)
axes[1].set_xlabel('Stations per event')
axes[1].set_title(f'Overfit subset (n={len(ov)})')

fig.suptitle('Number of Recording Stations per Event', fontsize=13)
fig.tight_layout()
fig.savefig(OUT_DIR / '04_stations_per_event.png')
plt.show()

## 7. 图 5 — Overfit 子集 PGA 分布 & PGA vs 震源距

In [ ]:
ov_stations = station_meta[station_meta['is_overfit']].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# PGA 直方图
axes[0].hist(ov_stations['log10_pga'], bins=40, color='steelblue',
             edgecolor='white', alpha=0.7)
axes[0].set_xlabel('log₁₀(PGA) [m/s²]')
axes[0].set_ylabel('Count')
axes[0].set_title('PGA Distribution (overfit subset)')

# PGA vs 震源距，按震级着色
ov_merged = ov_stations.merge(ov[['EVENT','Magnitude']], on='EVENT')
sc = axes[1].scatter(ov_merged['hypo_dist_km'], ov_merged['log10_pga'],
                     s=8, c=ov_merged['Magnitude'], cmap='YlOrRd',
                     vmin=5.0, vmax=7.0, alpha=0.6)
cbar = fig.colorbar(sc, ax=axes[1], shrink=0.8)
cbar.set_label('Magnitude')
axes[1].set_xlabel('Hypocentral distance (km)')
axes[1].set_ylabel('log₁₀(PGA) [m/s²]')
axes[1].set_title('PGA vs. Distance (overfit subset)')

fig.tight_layout()
fig.savefig(OUT_DIR / '05_pga_distribution.png')
plt.show()

## 8. 图 6 — 示例波形（overfit 子集中最大震级事件，最近 3 个台站）

In [ ]:
# 选择最大震级事件
biggest = ov.sort_values('Magnitude', ascending=False).iloc[0]
ev_name = biggest['EVENT']
# 找到 HDF5 中对应的 event ID (需要映射 event_xxxx → 实际 ID)
em_events = [x.decode() if isinstance(x, bytes) else x for x in em['EVENT'][()]]
ev_idx = em_events.index(ev_name)
# HDF5 中 data/ 下的 key 是时间戳格式
event_keys = list(f['data'].keys())

# 从 station_metadata 找到该事件的台站，按距离排序
ev_stations = station_meta[station_meta['EVENT'] == ev_name].copy()
ev_stations = ev_stations.sort_values('hypo_dist_km')
closest_3 = ev_stations.head(3)

# 需要把 EVENT 映射到 HDF5 data key
# EVENT 格式: event_00000, 对应 event_metadata 中的索引
ev_num = int(ev_name.split('_')[1])
hdf5_event_key = event_keys[ev_num] if ev_num < len(event_keys) else None

print(f'Event: {ev_name} (M{biggest["Magnitude"]:.1f}), HDF5 key: {hdf5_event_key}')
print(f'Closest 3 stations (dist: {closest_3["hypo_dist_km"].values.round(1)} km)')

In [ ]:
if hdf5_event_key is not None:
    grp = f['data'][hdf5_event_key]
    waveforms = grp['waveforms'][()]
    p_picks = grp['p_picks'][()]
    sampling_rate = 100.0  # Hz
    comp_names = ['NS', 'EW', 'UD']

    wave_idxs = closest_3['wave_idx'].values if 'wave_idx' in closest_3.columns else list(range(3))
    # wave_idx 来自 CSV，但 station_meta 没有 wave_idx，需要从 train_csv 获取
    ev_train = train_csv[train_csv['EVENT'] == ev_name]
    # 按 station_meta 排序匹配
    closest_codes = closest_3['station_code'].values
    station_codes_hdf5 = [x.decode() if isinstance(x, bytes) else x for x in grp['station_codes'][()]]
    
    fig, axes = plt.subplots(3, 3, figsize=(14, 7), sharex=True)
    
    for si, (_, srow) in enumerate(closest_3.iterrows()):
        code = srow['station_code'].strip()
        # 找到该台站在 HDF5 中的 wave_idx
        matching_idxs = [i for i, c in enumerate(station_codes_hdf5) if c.strip() == code]
        if not matching_idxs:
            print(f'Warning: station {code} not found in HDF5')
            continue
        widx = matching_idxs[0]
        wf = waveforms[widx]  # (T, 3)
        ppick = p_picks[widx]
        t = np.arange(wf.shape[0]) / sampling_rate
        t_pick = ppick / sampling_rate
        
        for ci in range(3):
            ax = axes[ci][si]
            ax.plot(t, wf[:, ci], linewidth=0.3, color='steelblue')
            ax.axvline(t_pick, color='red', linewidth=0.8, linestyle='--', alpha=0.7)
            if ci == 0:
                ax.set_title(f'{code} ({srow["hypo_dist_km"]:.0f} km)', fontsize=10)
            if si == 0:
                ax.set_ylabel(f'{comp_names[ci]}\n(m/s²)', fontsize=9)
            if ci == 2:
                ax.set_xlabel('Time (s)')
    
    fig.suptitle(f'Waveforms — M{biggest["Magnitude"]:.1f} event, 3 closest stations\n'
                 f'(red dashed = P-wave pick)', fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT_DIR / '06_example_waveforms.png')
    plt.show()
else:
    print('Could not map event to HDF5 key')

## 9. 图 7 — Overfit 子集事件总览表（适合直接贴 PPT）

In [ ]:
summary = ov[['EVENT','Magnitude','DEPTH','Latitude','Longitude']].copy()
summary['N_Stations'] = summary['EVENT'].map(stations_per_ov)
summary = summary.sort_values('Magnitude', ascending=False).reset_index(drop=True)
summary.index = summary.index + 1
summary.columns = ['Event ID', 'Mag', 'Depth (km)', 'Lat (°N)', 'Lon (°E)', '# Stations']

fig, ax = plt.subplots(figsize=(8, 5))
ax.axis('off')
table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.0, 1.4)
# 表头加粗
for j in range(len(summary.columns)):
    table[0, j].set_text_props(fontweight='bold')
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')
# 交替行色
for i in range(1, len(summary) + 1):
    color = '#D9E2F3' if i % 2 == 0 else 'white'
    for j in range(len(summary.columns)):
        table[i, j].set_facecolor(color)

ax.set_title('Overfit Subset — 16 Events', fontsize=13, fontweight='bold', pad=20)
fig.savefig(OUT_DIR / '07_event_summary_table.png')
plt.show()

## 10. 关闭 HDF5

In [ ]:
f.close()
print(f'\n所有图片已保存到 {OUT_DIR.resolve()}/')
print('图片列表:')
for p in sorted(OUT_DIR.glob('*.png')):
    print(f'  {p.name}')